In [1]:
from pathlib import Path
from datetime import datetime


import sys
import os
current_dir = os.getcwd()
project_root = os.path.join(current_dir, '..', '..') 

sys.path.append(project_root)
from src.models.xgboost_randomforrest_model_v2 import NFLModelV2
from src.models import hyper_parameter_tuning
from src.config import config_v2


c:\nfl_model\NFL_Model\nfl_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = NFLModelV2()
model.load_games(start_season=2022)
# Build offensive + defensive + differential features
model.build_feature_matrices(include_defense=True, include_differentials=True)
model.build_dataset()


Successfully connected to the database!


,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,total_points,binary_spread_label,...,opp_third_down_efficiency__def_roll5,opp_third_down_efficiency__diff_roll5,opp_third_down_efficiency__off_roll10,opp_third_down_efficiency__def_roll10,opp_third_down_efficiency__diff_roll10,opp_third_down_efficiency__sos_ratio,opp_third_down_efficiency__sos_inv_ratio,opp_third_down_efficiency__off_hist_z,opp_spread_feature,opp_over_under_feature
0,gs-202201BALNYJ,2022,1,NYJ,BAL,9,24,-15,33,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-6.5,44.0
1,gs-202201BUFLAR,2022,1,LAR,BUF,10,31,-21,41,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.5,52.0
2,gs-202201CLECAR,2022,1,CAR,CLE,24,26,-2,50,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,42.0
3,gs-202201DENSEA,2022,1,SEA,DEN,17,16,1,33,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-6.5,44.5
4,gs-202201GNBMIN,2022,1,MIN,GNB,23,7,16,30,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.5,46.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,gs-202504NYJMIA,2025,4,MIA,NYJ,27,21,6,48,1,...,0.484777,-0.158010,0.344409,0.419528,-0.075119,0.832809,1.200756,-1.485572,2.5,44.5
914,gs-202504PHITAM,2025,4,TAM,PHI,25,31,-6,56,1,...,0.331810,0.084124,0.352481,0.289730,0.062751,1.180994,0.846744,1.199540,-3.5,44.0
915,gs-202504SEAARI,2025,4,ARI,SEA,20,23,-3,43,1,...,0.335018,0.063377,0.377403,0.319533,0.057870,0.843588,1.185413,-0.590641,-1.5,43.0
916,gs-202504TENHOU,2025,4,HOU,TEN,26,0,26,26,1,...,0.374013,-0.027346,0.401526,0.382733,0.018793,0.958772,1.043000,-0.862288,7.0,39.5


In [3]:
q = f"""
SELECT
    g.gamesummaryid,
    g.season,
    g.week,
    g.hometeamid,
    g.awayteamid,
    g.homescore,
    g.awayscore,
    g.spread,
    g.spreadfavoriteteam,
    g.spreadteamcovered,
    g.over_under,
    g.overunderresults,
    ts.teamid,
    ts.time_of_possession,
    ts.cmp_att_yd_td_int,
    ts.fourth_down_conv,
    ts.rush_yds_tds,
    ts.fumbles_lost,
    ts.turnovers,
    ts.sacked_yards,
    ts.third_down_conv,
    ts.total_yards,
    ts.first_downs,
    ts.penalties_yards
FROM stats.gamesummary g
JOIN stats.teamstats ts ON ts.gamesummaryid = g.gamesummaryid
WHERE g.season >= {2024}
ORDER BY g.season, g.week, g.gamesummaryid;
"""
from src.utils.db_utils import execute_query
df = execute_query(q)
df

Successfully connected to the database!


,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,spreadfavoriteteam,spreadteamcovered,...,cmp_att_yd_td_int,fourth_down_conv,rush_yds_tds,fumbles_lost,turnovers,sacked_yards,third_down_conv,total_yards,first_downs,penalties_yards
0,gs-202401ARIBUF,2024,1,BUF,ARI,34,28,-6.5,BUF,BUF,...,21-31-162-1-0,0-1,25-124-1,1-1,1,4-16,7-13,270,18,5-31
1,gs-202401ARIBUF,2024,1,BUF,ARI,34,28,-6.5,BUF,BUF,...,18-23-232-2-0,2-2,33-130-2,1-1,1,2-10,3-9,352,23,9-65
2,gs-202401BALKAN,2024,1,KAN,BAL,27,20,-3.0,KAN,KAN,...,20-28-291-1-1,0-0,20-72-2,0-0,1,2-10,4-9,353,21,6-45
3,gs-202401BALKAN,2024,1,KAN,BAL,27,20,-3.0,KAN,KAN,...,26-41-273-1-0,1-2,32-185-1,1-1,1,1-6,7-14,452,25,7-64
4,gs-202401CARNOR,2024,1,NOR,CAR,47,10,-4.0,NOR,NOR,...,13-31-161-0-2,1-3,20-58-1,1-1,3,4-26,1-10,193,11,6-80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
693,gs-202504SEAARI,2025,4,ARI,SEA,20,23,-1.5,SEA,SEA,...,27-41-200-2-2,1-1,17-89-0,0-0,2,6-36,6-15,253,18,7-46
694,gs-202504TENHOU,2025,4,HOU,TEN,26,0,-7.0,HOU,HOU,...,22-28-233-2-0,3-3,35-129-1,0-0,0,2-9,6-15,353,20,6-50
695,gs-202504TENHOU,2025,4,HOU,TEN,26,0,-7.0,HOU,HOU,...,10-26-108-0-1,0-1,18-82-0,0-0,1,2-15,2-11,175,10,4-35
696,gs-202504WASATL,2025,4,ATL,WAS,34,27,-1.5,ATL,ATL,...,20-26-313-2-1,1-1,37-128-2,2-0,1,1-6,6-12,435,24,5-45


In [5]:
df["over_under"]

0      46.5
1      46.5
2      47.0
3      47.0
4      41.5
       ... 
693    43.0
694    39.5
695    39.5
696    43.5
697    43.5
Name: over_under, Length: 698, dtype: float64

In [5]:
model._dataset['rushing_efficiency__off_roll10']

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
         ...   
913    4.045699
914    5.191854
915    4.957516
916    4.696220
917    4.353786
Name: rushing_efficiency__off_roll10, Length: 918, dtype: float64

In [3]:
model.build_feature_matrices()

['turnovers__off_hist',
 'turnovers__def_hist',
 'turnovers__diff_hist',
 'turnovers__off_roll2',
 'turnovers__def_roll2',
 'turnovers__diff_roll2',
 'turnovers__off_roll5',
 'turnovers__def_roll5',
 'turnovers__diff_roll5',
 'turnovers__off_roll10',
 'turnovers__def_roll10',
 'turnovers__diff_roll10',
 'turnovers__sos_ratio',
 'turnovers__sos_inv_ratio',
 'turnovers__off_hist_z',
 'total_yards__off_hist',
 'total_yards__def_hist',
 'total_yards__diff_hist',
 'total_yards__off_roll2',
 'total_yards__def_roll2',
 'total_yards__diff_roll2',
 'total_yards__off_roll5',
 'total_yards__def_roll5',
 'total_yards__diff_roll5',
 'total_yards__off_roll10',
 'total_yards__def_roll10',
 'total_yards__diff_roll10',
 'total_yards__sos_ratio',
 'total_yards__sos_inv_ratio',
 'total_yards__off_hist_z',
 'pass_attempts__off_hist',
 'pass_attempts__def_hist',
 'pass_attempts__diff_hist',
 'pass_attempts__off_roll2',
 'pass_attempts__def_roll2',
 'pass_attempts__diff_roll2',
 'pass_attempts__off_roll5',


In [15]:
model.raw_data

,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,spreadfavoriteteam,spreadteamcovered,...,cmp_att_yd_td_int,fourth_down_conv,rush_yds_tds,fumbles_lost,turnovers,sacked_yards,third_down_conv,total_yards,first_downs,penalties_yards
0,gs-201001ARISTL,2010,1,STL,ARI,13,17,-3.0,ARI,ARI,...,22-41-297-1-0,0-0,21-112-1,7-4,4,2-31,5-13,378,21,10-72
1,gs-201001ARISTL,2010,1,STL,ARI,13,17,-3.0,ARI,ARI,...,32-55-253-1-3,2-3,24-85-0,2-1,4,2-13,8-20,325,20,5-40
2,gs-201001ATLPIT,2010,1,PIT,ATL,15,9,-1.5,ATL,PIT,...,27-44-252-0-1,0-0,25-58-0,0-0,1,2-15,6-16,295,18,3-24
3,gs-201001ATLPIT,2010,1,PIT,ATL,15,9,-1.5,ATL,PIT,...,18-26-236-0-1,0-0,31-143-1,0-0,1,3-25,4-14,354,14,4-25
4,gs-201001BALNYJ,2010,1,NYJ,BAL,9,10,-1.0,NYJ,Push,...,20-38-248-0-1,0-0,35-49-1,2-2,3,2-15,11-19,282,20,5-38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8245,gs-202503NORSEA,2025,3,SEA,NOR,44,13,-7.0,SEA,SEA,...,16-21-233-2-0,0-0,33-87-2,1-1,1,0-0,3-9,320,22,8-70
8246,gs-202503NYJTAM,2025,3,TAM,NYJ,29,27,-6.5,TAM,TAM,...,19-29-233-1-0,0-0,34-122-0,1-0,0,1-8,4-13,347,19,14-124
8247,gs-202503NYJTAM,2025,3,TAM,NYJ,29,27,-6.5,TAM,TAM,...,26-36-197-2-1,1-3,23-99-0,1-1,2,4-29,3-11,267,21,7-81
8248,gs-202503PITNWE,2025,3,NWE,PIT,14,21,-1.5,PIT,PIT,...,16-23-139-2-1,0-0,26-64-1,1-0,1,0-0,4-9,203,17,8-59


In [16]:
import pandas as pd
import numpy as np

np.where((model.raw_data["hometeamid"].values==model.raw_data["spreadfavoriteteam"].values), model.raw_data["spread"].values*-1, model.raw_data["spread"])

array([-3. , -3. , -1.5, ...,  6.5, -1.5, -1.5])

In [17]:
df=model.raw_data.set_index("gamesummaryid")

In [18]:
df

,season,week,hometeamid,awayteamid,homescore,awayscore,spread,spreadfavoriteteam,spreadteamcovered,over_under,...,cmp_att_yd_td_int,fourth_down_conv,rush_yds_tds,fumbles_lost,turnovers,sacked_yards,third_down_conv,total_yards,first_downs,penalties_yards
gamesummaryid,,,,,,,,,,,,,,,,,,,,,
gs-201001ARISTL,2010,1,STL,ARI,13,17,-3.0,ARI,ARI,39.5,...,22-41-297-1-0,0-0,21-112-1,7-4,4,2-31,5-13,378,21,10-72
gs-201001ARISTL,2010,1,STL,ARI,13,17,-3.0,ARI,ARI,39.5,...,32-55-253-1-3,2-3,24-85-0,2-1,4,2-13,8-20,325,20,5-40
gs-201001ATLPIT,2010,1,PIT,ATL,15,9,-1.5,ATL,PIT,39.5,...,27-44-252-0-1,0-0,25-58-0,0-0,1,2-15,6-16,295,18,3-24
gs-201001ATLPIT,2010,1,PIT,ATL,15,9,-1.5,ATL,PIT,39.5,...,18-26-236-0-1,0-0,31-143-1,0-0,1,3-25,4-14,354,14,4-25
gs-201001BALNYJ,2010,1,NYJ,BAL,9,10,-1.0,NYJ,Push,36.5,...,20-38-248-0-1,0-0,35-49-1,2-2,3,2-15,11-19,282,20,5-38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
gs-202503NORSEA,2025,3,SEA,NOR,44,13,-7.0,SEA,SEA,41.5,...,16-21-233-2-0,0-0,33-87-2,1-1,1,0-0,3-9,320,22,8-70
gs-202503NYJTAM,2025,3,TAM,NYJ,29,27,-6.5,TAM,TAM,43.0,...,19-29-233-1-0,0-0,34-122-0,1-0,0,1-8,4-13,347,19,14-124
gs-202503NYJTAM,2025,3,TAM,NYJ,29,27,-6.5,TAM,TAM,43.0,...,26-36-197-2-1,1-3,23-99-0,1-1,2,4-29,3-11,267,21,7-81
